In [2]:
import os, time, json, requests
import pandas as pd

URL = "https://merpis.ptmerahputih.com/api/vessel"
TOKEN = "Bearer 7|pLRWvS5Yl7Lob0WoU1IDHoT99bnPXBuz9GUnEriy692b2fe1"
headers = {"Accept": "application/json", "Authorization": f"Bearer {TOKEN}"}

t0 = time.time()
r = requests.get(URL, headers=headers, timeout=(10, 600))
print("Status:", r.status_code, "|", f"{time.time()-t0:.1f}s", "|", f"{len(r.content)/1e6:.3f} MB")
r.raise_for_status()
data = r.json()

with open("vessel_raw.json", "w", encoding="utf-8") as f:
    json.dump(data, f, ensure_ascii=False)

print("Tipe:", type(data).__name__)
if isinstance(data, dict):
    print({k: v for k, v in data.items() if not isinstance(v, (list, dict))})

Status: 200 | 0.4s | 0.025 MB
Tipe: dict
{'code': 200, 'message': 'success'}


In [3]:
# Cari list of dict di level mana pun
def cari_list(obj, path="root"):
    if isinstance(obj, list) and obj and isinstance(obj[0], dict):
        yield path, obj
    elif isinstance(obj, dict):
        for k, v in obj.items():
            yield from cari_list(v, f"{path}.{k}")

kandidat = list(cari_list(data))
for path, lst in kandidat:
    print(f"{path:30s} {len(lst):6d} item | key: {list(lst[0].keys())[:10]}")

path, records = max(kandidat, key=lambda x: len(x[1]))
df_v = pd.json_normalize(records)
print("\nDipakai:", path, "| Shape:", df_v.shape)
if "id" in df_v.columns:
    print("id unik:", df_v["id"].nunique(), "| min/max:", df_v["id"].min(), "/", df_v["id"].max())

root.data.vessel                   84 item | key: ['id', 'name', 'type', 'created_at', 'updated_at', 'active', 'owner', 'hp', 'builtyear', 'speedempty']

Dipakai: root.data.vessel | Shape: (84, 16)
id unik: 84 | min/max: 1 / 84


In [4]:
# Ringkasan kolom + kolom nested yang belum terurai
pd.set_option("display.max_rows", None, "display.max_colwidth", 60, "display.width", 200)
display(pd.DataFrame({
    "kolom": df_v.columns,
    "tipe": df_v.dtypes.astype(str).values,
    "terisi": df_v.notna().sum().values,
    "kosong_%": (df_v.isna().mean() * 100).round(1).values,
    "unik": df_v.nunique(dropna=True).values,
    "contoh": [df_v[c].dropna().iloc[0] if df_v[c].notna().any() else None for c in df_v.columns],
}))

for c in df_v.columns:
    if df_v[c].apply(lambda x: isinstance(x, (list, dict))).any():
        print("Nested belum terurai:", c)

df_v.head()

,kolom,tipe,terisi,kosong_%,unik,contoh
0,id,int64,84,0.0,84,1
1,name,object,84,0.0,83,TB. ABBA 01
2,type,int64,84,0.0,2,1
3,created_at,object,84,0.0,12,2026-04-08T05:38:18.000000Z
4,updated_at,object,84,0.0,24,2026-04-08T05:38:18.000000Z
5,active,int64,84,0.0,1,1
6,owner,object,84,0.0,5,PMP
7,hp,object,44,47.6,8,2000
8,builtyear,object,81,3.6,41,2021-04-23
9,speedempty,float64,42,50.0,4,6.0


,id,name,type,created_at,updated_at,active,owner,hp,builtyear,speedempty,speedload,feet,code,docking,dockingdate,sapb1
0,1,TB. ABBA 01,1,2026-04-08T05:38:18.000000Z,2026-04-08T05:38:18.000000Z,1,PMP,2000,2021-04-23,6.0,4.0,None,TB.AB01,0,None,PMP_LIVE
1,2,TB. DMP 01,1,2026-04-08T05:38:18.000000Z,2026-04-08T05:38:18.000000Z,1,DMP,2000,2023-11-09,6.0,4.0,None,TB.DM01,0,None,DMP_LIVE
2,3,TB. DMP 03,1,2026-04-08T05:38:18.000000Z,2026-07-05T06:20:19.000000Z,1,DMP,2000,2023-11-09,6.0,4.0,None,TB.DM03,0,None,DMP_LIVE
3,4,TB. DMP 05,1,2026-04-08T05:38:18.000000Z,2026-04-08T05:38:18.000000Z,1,DMP,2000,2023-11-09,6.0,4.0,None,TB.DM05,0,None,DMP_LIVE
4,5,TB. DMP 07,1,2026-04-08T05:38:18.000000Z,2026-04-08T05:38:18.000000Z,1,DMP,2000,2023-12-14,6.0,4.0,None,TB.DM07,0,None,DMP_LIVE


In [5]:
import os, json, requests
import numpy as np
import pandas as pd

# 1. Ambil data mentah: pakai `data` kalau sudah ada, kalau tidak baca file, kalau tidak tarik ulang
try:
    data
except NameError:
    if os.path.exists("vessel_raw.json"):
        data = json.load(open("vessel_raw.json", encoding="utf-8"))
    else:
        TOKEN = os.environ.get("MERPIS_TOKEN_VESSEL") or os.environ.get("MERPIS_TOKEN_PROJECTS") or "ISI_TOKEN_BEARER"
        r = requests.get("https://merpis.ptmerahputih.com/api/vessel",
                         headers={"Accept": "application/json", "Authorization": f"Bearer {TOKEN}"},
                         timeout=(10, 600))
        r.raise_for_status()
        data = r.json()

# 2. Cari list of dict terbesar, lalu ratakan
def cari_list(obj, path="root"):
    if isinstance(obj, list) and obj and isinstance(obj[0], dict):
        yield path, obj
    elif isinstance(obj, dict):
        for k, v in obj.items():
            yield from cari_list(v, f"{path}.{k}")

path, records = max(cari_list(data), key=lambda x: len(x[1]))
print("Sumber list:", path, "|", len(records), "item")

df_raw = pd.json_normalize(records)

# 3. Kolom berisi list/dict yang belum terurai: laporkan ukurannya, lalu pisahkan dari tabel utama
nested = [c for c in df_raw.columns if df_raw[c].apply(lambda x: isinstance(x, (list, dict))).any()]
for c in nested:
    mb = df_raw[c].apply(lambda x: len(json.dumps(x, default=str)) if isinstance(x, (list, dict)) else 0).sum() / 1e6
    print(f"Nested: {c:40s} {mb:.2f} MB")
d = df_raw.drop(columns=nested).copy()

Sumber list: root.data.vessel | 84 item


In [6]:
# 4. Cleansing
d.columns = [c.strip().lower().replace(".", "_").replace(" ", "_") for c in d.columns]

# String: trim, kosong/None/null -> NaN
obj_cols = d.select_dtypes(include=["object", "string"]).columns
for c in obj_cols:
    d[c] = d[c].apply(lambda x: x.strip() if isinstance(x, str) else x)
    d[c] = d[c].replace({"": np.nan, "None": np.nan, "none": np.nan, "null": np.nan, "NaN": np.nan})

# Waktu: *_at (UTC) -> WIB tanpa timezone; kolom tanggal lain -> datetime bila >=90% terbaca
for c in d.columns:
    if d[c].dtype != "object":
        continue
    if c.endswith("_at"):
        s = pd.to_datetime(d[c], errors="coerce", utc=True)
        if s.notna().sum() >= d[c].notna().sum() * 0.9:
            d[c] = s.dt.tz_convert("Asia/Jakarta").dt.tz_localize(None)
            continue
    if "date" in c or c.endswith(("builtyear", "_at")):
        s = pd.to_datetime(d[c], errors="coerce")
        if d[c].notna().any() and s.notna().sum() >= d[c].notna().sum() * 0.9:
            d[c] = s
            continue

# Numerik: hanya jika SEMUA nilai terisi bisa dikonversi dan tidak ada angka berawalan 0 (kode/telepon)
for c in d.columns:
    if d[c].dtype != "object" or not d[c].notna().any():
        continue
    s = d[c].dropna().astype(str)
    if s.str.match(r"^0\d+").any():
        continue
    conv = pd.to_numeric(d[c], errors="coerce")
    if conv.notna().sum() == d[c].notna().sum():
        d[c] = conv

# Duplikat
if "id" in d.columns:
    print("Duplikat id:", d["id"].duplicated().sum())
    d = d.drop_duplicates(subset="id", keep="first").sort_values("id", ascending=False)
d = d.reset_index(drop=True)

print("Shape:", df_raw.shape, "->", d.shape)

Duplikat id: 0
Shape: (84, 16) -> (84, 16)


In [7]:
# 5. Ringkasan + kandidat kolom yang bisa dibuang (hanya dilaporkan, tidak dibuang)
summary = pd.DataFrame({
    "kolom": d.columns,
    "tipe": d.dtypes.astype(str).values,
    "terisi": d.notna().sum().values,
    "kosong_%": (d.isna().mean() * 100).round(1).values,
    "unik": d.nunique().values,
    "contoh": [d[c].dropna().iloc[0] if d[c].notna().any() else None for c in d.columns],
})
pd.set_option("display.max_rows", None, "display.max_colwidth", 50, "display.width", 200)
display(summary)

print("Kosong semua  :", d.columns[d.isna().all()].tolist())
print("Konstan       :", [c for c in d.columns if d[c].nunique(dropna=False) == 1 and c != "id"])

# 6. Export untuk dicek
with pd.ExcelWriter("vessel_cleansing_check.xlsx", engine="openpyxl") as w:
    d.to_excel(w, sheet_name="data_clean", index=False)
    summary.to_excel(w, sheet_name="ringkasan_kolom", index=False)
    for ws in w.sheets.values():
        ws.freeze_panes = "B2"
        ws.auto_filter.ref = ws.dimensions

print("Tersimpan: vessel_cleansing_check.xlsx |", d.shape[0], "baris x", d.shape[1], "kolom")

,kolom,tipe,terisi,kosong_%,unik,contoh
0,id,int64,84,0.0,84,84
1,name,object,84,0.0,83,BG. BAR BERSAUDARA 10
2,type,int64,84,0.0,2,2
3,created_at,datetime64[ns],84,0.0,12,2026-11-09 10:10:39
4,updated_at,datetime64[ns],84,0.0,24,2026-11-09 10:11:00
5,active,int64,84,0.0,1,1
6,owner,object,84,0.0,5,PMP
7,hp,object,44,47.6,8,1031
8,builtyear,datetime64[ns],81,3.6,41,2026-07-01 00:00:00
9,speedempty,float64,42,50.0,4,6.0


Kosong semua  : ['dockingdate']
Konstan       : ['active', 'docking', 'dockingdate']
Tersimpan: vessel_cleansing_check.xlsx | 84 baris x 16 kolom
